In [1]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
a2a-sdk 0.3.22 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requir

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
%env TOKENIZERS_PARALLELISM=false

env: TOKENIZERS_PARALLELISM=false


In [4]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import seaborn as sns
import tokenize
import torch
import torch.nn as nn
import torch.optim as optim
import transformers

from datasets import Dataset

from math import ceil

from scipy.special import softmax

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    logging
)

from torch.utils.data import DataLoader

from tqdm.auto import tqdm

2026-01-18 14:14:41.555298: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768745681.706699      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768745681.753366      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768745682.118162      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768745682.118197      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768745682.118200      55 computation_placer.cc:177] computation placer alr

(OBS) Code block added to delete the SQL status of the checkpoint (it has 7 GBs, so it ocuppies much space on the output). Add this code block at the beginning of the script, right after the imports.

In [5]:
path = "/kaggle/working/state.db"

if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")


state.db not found


In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

In [7]:
base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-b/Task_B"

training_path = base_path + "/train.parquet"
validation_path = base_path + "/validation.parquet"
test_path = base_path + "/test.parquet"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_df = pd.read_parquet(test_path)

In [8]:
llm_df = training_df[training_df["generator"].str.lower() != "human"]
human_df = training_df[training_df["generator"].str.lower() == "human"]

TARGET_HUMAN_RATIO = 0.7

num_llm = len(llm_df)
num_human = int(num_llm * TARGET_HUMAN_RATIO / (1 - TARGET_HUMAN_RATIO))

human_df = human_df.sample(
    n=min(len(human_df), num_human),
    random_state=1337
)

training_df = pd.concat([llm_df, human_df]).sample(frac=1, random_state=1337)

In [9]:
training_df.to_parquet("training_sample_set.parquet", index=False)
validation_df.to_parquet("validation_sample_set.parquet", index=False)

In [10]:
pretrained_model = "microsoft/unixcoder-base"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
counts = training_df["label"].value_counts().sort_index()

weights = 1.0 / torch.sqrt(
    torch.tensor(counts.values, dtype=torch.float)
)
weights = weights / weights.mean()

class_weights = weights.to(device)

The functions below preprocess code samples and erase the comments.

In [14]:
def is_escaped(result, i):
    count = 0
    i -= 1
    while i >= 0 and result[i] == '\\':
        count += 1
        i -= 1
    return count % 2 == 1

In [15]:
def strip_jcg_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    in_single_line_comment = False
    in_multi_line_comment = False
    string_delimiter = None
    in_verbatim_string = False

    while i < n:
        c = code[i]
        next_c = code[i + 1] if i + 1 < n else ''

        if in_single_line_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_single_line_comment = False
            i += 1
            continue

        if in_multi_line_comment:
            if c == '*' and next_c == '/':
                result[i] = result[i + 1] = ' '
                in_multi_line_comment = False
                i += 2
            else:
                if c != '\n':
                    result[i] = ' '
                i += 1
            continue

        if string_delimiter is not None:
            if string_delimiter == '`':
                if c == '`':
                    string_delimiter = None
            elif string_delimiter == '"':
                if c == '"' and not is_escaped(code, i):
                    string_delimiter = None
            elif string_delimiter == "'":
                if c == "'" and not is_escaped(code, i):
                    string_delimiter = None
            i += 1
            continue

        if in_verbatim_string:
            if c == '"' and next_c == '"':
                i += 2
            else:
                if c == '"' and next_c != '"':
                    in_verbatim_string = False
                i += 1
            continue

        if c == '@' and next_c == '"':
            in_verbatim_string = True
            i += 2
            continue

        if c == "'" or c == '"' or c == '`':
            string_delimiter = c
            i += 1
            continue

        if c == '/' and next_c == '/':
            result[i] = result[i + 1] = ' '
            in_single_line_comment = True
            i += 2
            continue

        if c == '/' and next_c == '*':
            result[i] = result[i + 1] = ' '
            in_multi_line_comment = True
            i += 2
            continue

        i += 1

    return ''.join(result)

In [16]:
def strip_php_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    in_single_line_comment = False
    in_multi_line_comment = False
    string_delimiter = None
    in_heredoc = False
    heredoc_id = None

    while i < n:
        c = code[i]
        next_c = code[i + 1] if i + 1 < n else ''

        if i == 0 or code[i - 1] == '\n':
            line_start = i
        else:
            line_start = None

        if in_heredoc:
            if line_start is not None:
                j = line_start
                k = 0
                while j < n and k < len(heredoc_id) and code[j] == heredoc_id[k]:
                    j += 1
                    k += 1
                if k == len(heredoc_id) and (j == n or code[j] in (';', '\n')):
                    in_heredoc = False
                    heredoc_id = None
            i += 1
            continue

        if in_single_line_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_single_line_comment = False
            i += 1
            continue

        if in_multi_line_comment:
            if c == '*' and next_c == '/':
                result[i] = result[i + 1] = ' '
                in_multi_line_comment = False
                i += 2
            else:
                if c != '\n':
                    result[i] = ' '
                i += 1
            continue

        if string_delimiter is not None:
            if c == string_delimiter and not is_escaped(code, i):
                string_delimiter = None
            i += 1
            continue

        if c == '<' and code[i:i+3] == '<<<':
            j = i + 3

            while j < n and code[j].isspace():
                j += 1

            if j < n and code[j] in ("'", '"'):
                quote = code[j]
                j += 1
                start = j
                while j < n and code[j] != quote:
                    j += 1
                heredoc_id = code[start:j]
                j += 1
            else:
                start = j
                while j < n and (code[j].isalnum() or code[j] == '_'):
                    j += 1
                heredoc_id = code[start:j]

            in_heredoc = True
            i = j
            continue


        if c == "'" or c == '"':
            string_delimiter = c
            i += 1
            continue

        if c == '/' and next_c == '/':
            result[i] = result[i + 1] = ' '
            in_single_line_comment = True
            i += 2
            continue

        if c == '#':
            result[i] = ' '
            in_single_line_comment = True
            i += 1
            continue

        if c == '/' and next_c == '*':
            result[i] = result[i + 1] = ' '
            in_multi_line_comment = True
            i += 2
            continue

        i += 1

    return ''.join(result)

In [17]:
def strip_python_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    string_delimiter = None
    in_comment = False

    while i < n:
        c = code[i]

        if in_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_comment = False
            i += 1
            continue

        if string_delimiter is not None:
            if c == string_delimiter and not is_escaped(code, i):
                string_delimiter = None
            i += 1
            continue

        if c in ("'", '"'):
            string_delimiter = c
            i += 1
            continue

        if c == '#':
            result[i] = ' '
            in_comment = True
            i += 1
            continue

        i += 1

    return ''.join(result)

In [18]:
fallback_count = 0

def remove_python_docstrings(code: str) -> str:
    global fallback_count
    try:
        tokens = tokenize.generate_tokens(io.StringIO(code).readline)
        result = []
    
        scope_stack = [True]
    
        for tok in tokens:
            tok_type, tok_str, _, _, _ = tok
    
            if tok_type == tokenize.INDENT:
                scope_stack.append(True)
            elif tok_type == tokenize.DEDENT:
                scope_stack.pop()
            elif tok_type == tokenize.STRING and scope_stack[-1]:
                scope_stack[-1] = False
                continue
            elif tok_type not in (tokenize.NL, tokenize.NEWLINE):
                scope_stack[-1] = False
    
            result.append(tok)
    
        return tokenize.untokenize(result)

    except (IndentationError, SyntaxError, tokenize.TokenError):
        fallback_count += 1
        return code

In [19]:
def strip_python_comments_and_docstrings(code: str) -> str:
    code = strip_python_comments(code)
    code = remove_python_docstrings(code)
    return code

In [20]:
def normalize_whitespace(code: str) -> str:
    lines = code.splitlines()
    normalized = []

    for line in lines:
        stripped = line.rstrip()

        if stripped:
            m = re.match(r'^(\s*)(.*)$', stripped)
            indent, content = m.groups()
            content = re.sub(r' {2,}', ' ', content)
            normalized.append(indent + content)

    return '\n'.join(normalized)

In [21]:
def remove_jcg_comments(code: str) -> str:
    return normalize_whitespace(strip_jcg_comments(code))


def remove_php_comments(code: str) -> str:
    return normalize_whitespace(strip_php_comments(code))


def remove_python_comments(code: str) -> str:
    return normalize_whitespace(strip_python_comments_and_docstrings(code))

In [22]:
def clean_code(code: str, language: str) -> str:
    lang = language.lower() if isinstance(language, str) else ""
    if lang == "python":
        return remove_python_comments(code)
    elif lang == "php":
        return remove_php_comments(code)
    else:
        return remove_jcg_comments(code)

In [23]:
def preprocess_function(examples: pd.DataFrame):
    # cleaned_code = [
    #     clean_code(code, lang)
    #     for code, lang in zip(examples["code"], examples["language"])
    # ]

    # examples["code"] = cleaned_code

    tokenized = tokenizer(
        examples["code"],
        truncation=True,
        max_length=192
    )

    return tokenized

(OBS) It is good practice to set the format to torch after you tokenize the datasets. Add the .set_format("torch") instructions right after you tokenized the datasets.

In [24]:
training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True)
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True)

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")

Map:   0%|          | 0/193013 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [25]:
print("Docstring fallback count:", fallback_count)

Docstring fallback count: 0


In [26]:
id2label = {
  "0": "human",
  "1": "deepseek",
  "2": "qwen",
  "3": "01-ai",
  "4": "bigcode",
  "5": "gemma",
  "6": "phi",
  "7": "meta-llama",
  "8": "ibm-granite",
  "9": "mistral",
  "10": "openai"
}


label2id = {
  "human": "0",
  "deepseek": "1",
  "qwen": "2",
  "01-ai": "3",
  "bigcode": "4",
  "gemma": "5",
  "phi": "6",
  "meta-llama": "7",
  "ibm-granite": "8",
  "mistral": "9",
  "openai": "10"
}

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model,
    num_labels=11,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
model.to(device)
print("\nRunning on device:", device)


Running on device: cuda


In [29]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

1
Tesla T4


In [30]:
logging.set_verbosity_info()

In [31]:
class TrainingProgressCallback(transformers.TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            tqdm.write(f"Step {state.global_step}/{state.max_steps}")

(OBS) In the TrainingArguments parameters, you have to set the following:
- output_dir="/kaggle/working/checkpoints" (this is the Kaggle notebook folder that stores the output files; you will need to have persistent data, to achieve that the checkpoints must be saved in this folder);
- num_train_epochs=3;
- load_best_model_at_end=True (for best results);
- eval_strategy="epoch";
- save_strategy="epoch";
- save_total_limit=2 (or 3, you set here the last number of checkpoints that remain saved);
- logging_dir="/kaggle/working/logs" (optional, stores log information);
- logging_steps=3000 (optional, add only if you put logging_dir too);

In [32]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints_b",
    seed=42,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.02,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    label_smoothing_factor=0.0,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=0,
    report_to=[]
)

PyTorch: setting up devices


In [33]:
class OptimizedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            label_smoothing=0.05
        )

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [34]:
trainer = OptimizedTrainer(
    model=model,
    args=training_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

/tmp/ipykernel_55/92150513.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `OptimizedTrainer.__init__`. Use `processing_class` instead.
  trainer = OptimizedTrainer(


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Using auto half precision backend


(OBS) The first execution will have just trainer.train(), because initially you won't have any checkpoints, you start from 0 with the training. After that, you need to replace this instruction with trainer.train(resume_from_checkpoint=True), such that it resumes the training from the last checkpoint.

In [35]:
trainer.train()

The following columns in the Training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: generator, __index_level_0__, language, code. If generator, __index_level_0__, language, code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 193,013
  Num Epochs = 4
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 4
  Total optimization steps = 12,064
  Number of trainable parameters = 125,938,187


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision,Recall
1,1.576100,1.516606,0.878003,0.855820,0.912648,0.855820
2,1.452300,1.462813,0.889016,0.867990,0.920951,0.867990
3,1.362800,1.459580,0.894202,0.874080,0.923107,0.874080
4,1.300600,1.442669,0.900338,0.883180,0.924775,0.883180


The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: generator, language, code. If generator, language, code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 100000
  Batch size = 32
Saving model checkpoint to /kaggle/working/checkpoints_b/checkpoint-3016
Configuration saved in /kaggle/working/checkpoints_b/checkpoint-3016/config.json
Model weights saved in /kaggle/working/checkpoints_b/checkpoint-3016/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_b/checkpoint-3016/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_b/checkpoint-3016/special_tokens_map.json
Deleting older checkpoint [/kaggle/working/checkpoints_b/checkpoint-9048] due to args.save_total_limit
The following columns in the Evaluation set don't have a corresponding argumen

TrainOutput(global_step=12064, training_loss=1.4860317700737666, metrics={'train_runtime': 9437.6335, 'train_samples_per_second': 81.806, 'train_steps_per_second': 1.278, 'total_flos': 7.61819367567191e+16, 'train_loss': 1.4860317700737666, 'epoch': 4.0})

In [36]:
trainer.save_model("/kaggle/working/final_model")
tokenizer.save_pretrained("/kaggle/working/final_tokenizer")

Saving model checkpoint to /kaggle/working/final_model
Configuration saved in /kaggle/working/final_model/config.json
Model weights saved in /kaggle/working/final_model/model.safetensors
tokenizer config file saved in /kaggle/working/final_model/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_model/special_tokens_map.json
tokenizer config file saved in /kaggle/working/final_tokenizer/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_tokenizer/special_tokens_map.json


('/kaggle/working/final_tokenizer/tokenizer_config.json',
 '/kaggle/working/final_tokenizer/special_tokens_map.json',
 '/kaggle/working/final_tokenizer/vocab.json',
 '/kaggle/working/final_tokenizer/merges.txt',
 '/kaggle/working/final_tokenizer/added_tokens.json',
 '/kaggle/working/final_tokenizer/tokenizer.json')

In [37]:
# tokenizer = AutoTokenizer.from_pretrained(
#     "/kaggle/working/final_tokenizer"
# )
# model = AutoModelForSequenceClassification.from_pretrained(
#     "/kaggle/working/final_model"
# )

In [38]:
def tokenize_test_set(examples: pd.DataFrame):
    return tokenizer(
        text_target=examples["code"],
        truncation=True,
        max_length=192
    )

In [39]:
test_dataset = Dataset.from_pandas(test_df)

test_tokenized_set = test_dataset.map(tokenize_test_set, batched=True)

test_tokenized_set.set_format("torch")

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

In [40]:
test_logits = torch.tensor(
    trainer.predict(test_tokenized_set).predictions
)

probs = softmax(test_logits, axis=1)

CONFIDENCE_THRESHOLD = 0.35

preds = []
for p in probs:
    if p.max() < CONFIDENCE_THRESHOLD:
        preds.append(p.argmax())
    else:
        preds.append(p.argmax())

preds = np.array(preds)

The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: ID, __index_level_0__, code. If ID, __index_level_0__, code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 500000
  Batch size = 32


In [42]:
pd.DataFrame({
    "ID": test_df["ID"],
    "label": preds
}).to_csv("Predictions.csv", index=False)

In [ ]:
# def stream_predict(
#     model,
#     dataset,
#     tokenizer,
#     batch_size,
#     num_workers,
#     device,
# ):
#     model.eval()
#     model.to(device)

#     data_collator = DataCollatorWithPadding(
#         tokenizer=tokenizer,
#         return_tensors="pt"
#     )

#     dataloader = DataLoader(
#         dataset,
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=num_workers,
#         pin_memory=True,
#         collate_fn=data_collator
#     )

#     all_preds = []

#     with torch.no_grad():
#         for batch in tqdm(dataloader, desc="Streaming inference"):
#             batch = {k: v.to(device) for k, v in batch.items()}

#             outputs = model(**batch)
#             preds = torch.argmax(outputs.logits, dim=-1)

#             all_preds.append(preds.cpu())

#     return torch.cat(all_preds).numpy()

In [ ]:
# test_tokenized_set_cleaned = test_tokenized_set.remove_columns(
#     [c for c in test_tokenized_set.column_names
#      if c not in ("input_ids", "attention_mask")]
# )

# print(test_tokenized_set_cleaned.column_names)

In [ ]:
# predictions = stream_predict(
#     model=model,
#     dataset=test_tokenized_set_cleaned,
#     tokenizer=tokenizer,
#     batch_size=256,
#     num_workers=0,
#     device=device
# )